In [ ]:
def CNN_model(input_data, noOfNeuron, learningRate, filterSize, numOfFilters, stride):
    model = nn.Sequential(
        nn.Conv2d(input_data.shape[3], numOfFilters, kernel_size=(filterSize, filterSize), stride=(stride, stride), padding=(filterSize//2, filterSize//2)),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2)),
        nn.Conv2d(numOfFilters, numOfFilters, kernel_size=(filterSize, filterSize), stride=(stride, stride), padding=(filterSize//2, filterSize//2)),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2)),
        nn.Flatten()
    )

    model = model.to(DEVICE)

    with torch.no_grad():
        sample_data = torch.zeros(1, input_data.shape[3], input_data.shape[1], input_data.shape[2]).to(DEVICE)
        flattened_size = model(sample_data).shape[1]

    model = nn.Sequential(
        nn.Conv2d(input_data.shape[3], numOfFilters, kernel_size=(filterSize, filterSize), stride=(stride, stride), padding=(filterSize//2, filterSize//2)),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2)),
        nn.Conv2d(numOfFilters, numOfFilters, kernel_size=(filterSize, filterSize), stride=(stride, stride), padding=(filterSize//2, filterSize//2)),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2)),
        nn.Flatten(),
        nn.Linear(flattened_size, noOfNeuron),
        nn.ReLU(),
        nn.Linear(noOfNeuron, 2)
    )

    model = model.to(DEVICE)
    return model


In [ ]:
# Initialize a count value to store the performance of each model
cnt = 0

# Convert training and test data to tensors
TrainData_tensor = torch.tensor(np.transpose(TrainData, (0, 3, 1, 2)), dtype=torch.float32).to(DEVICE)
TestData_tensor = torch.tensor(np.transpose(TestData, (0, 3, 1, 2)), dtype=torch.float32).to(DEVICE)
TrainLabel_tensor = torch.tensor(np.argmax(TrainLabel, axis=1), dtype=torch.long).to(DEVICE)
TestLabel_tensor = torch.tensor(np.argmax(TestLabel, axis=1), dtype=torch.long).to(DEVICE)

# Iterate through all possible combinations of filter size, filter number, and stride
for temp_FiltS in param_FiltS:          # Select each filter size in the list
    for temp_FiltN in param_FiltN:      # Select each filter number in the list
        for temp_Strid in param_Strid:  # Select each stride value in the list

            # Create, train, and validate a temporary CNN model with the current combination of hyperparameters
            temp_cnn_model = CNN_model(TrainData, noOfNeuron, learningRate, temp_FiltS, temp_FiltN, temp_Strid)
            criterion = nn.CrossEntropyLoss()
            optimizer = torch.optim.Adam(temp_cnn_model.parameters(), lr=learningRate)

            temp_cnn_model.train()
            for _ in range(Epoch):
                optimizer.zero_grad()
                Output = temp_cnn_model(TrainData_tensor)
                Loss = criterion(Output, TrainLabel_tensor)
                Loss.backward()
                optimizer.step()

            temp_cnn_model.eval()
            with torch.no_grad():
                Output_test = temp_cnn_model(TestData_tensor)
                Loss = criterion(Output_test, TestLabel_tensor).item()
                Accuracy = (torch.argmax(Output_test, dim=1) == TestLabel_tensor).float().mean().item()

            # Save the temporary model to a file with a corresponding name
            temp_cnn_model_name = f'CNN_FS{temp_FiltS}_FN{temp_FiltN}_St{temp_Strid}.h5'
            torch.save(temp_cnn_model.state_dict(), '/content/drive/MyDrive/Colab Notebooks/SavedFiles/ML_Models/GridSearch_CNN/' + temp_cnn_model_name)

            # Store the performance (accuracy) of the temporary model in the dataframe
            Accuracy_df.iloc[cnt, :] = [temp_FiltS, temp_FiltN, temp_Strid, Accuracy]
            cnt += 1

# Display the resulting dataframe with model performances
Accuracy_df
